# Day 3 — Control flow: conditionals, truthiness, exceptions
Objectives:
- if/elif/else, while/for.
- try/except/else/finally.
- Define and raise custom exceptions.

<!-- BEGIN BEGINNER NOTEBOOK DEEP DIVE -->
## How to use this notebook

This is the editable learner artifact for `python-03`. Read
`python/ds-60day/companion-guides/day03_control_flow_exceptions.md` first, then work here with the **Python (ds60sqlpy)**
kernel. Restart the kernel and run from top to bottom so an earlier
hidden value cannot make later code appear correct.

For every example: (1) write a prediction, (2) run the cell,
(3) compare the exact value, type, shape, rows, or side effect with
the stated observation, and (4) explain one mismatch before moving
on. For every exercise, use its dedicated work cell and include a
real assertion or bounded inspection. The official solution stays
closed until you have a tested attempt.

The notebook is deliberately offline after course setup. Do not add
`%pip`, credentials, absolute developer paths, or shell-specific
setup here. If an import fails, use the repository doctor and the
catalog from a terminal rather than changing only this kernel.

## Core mental model

Control flow is the route execution takes through a program. An
`if`/`elif`/`else` chain chooses exactly one branch: Python tests from
top to bottom and stops at the first true condition. Put narrow or more
specific rules before broader ones, and validate impossible inputs
before classifying valid ones.

Exceptions separate a normal return path from a failure path. A `try`
block should contain only the operation expected to fail, while
`except` names the failure the current layer can interpret. Catching
every exception hides programming bugs. `else` runs after a successful
`try`; `finally` runs whether success or failure occurred.

### Vocabulary

- **condition:** an expression evaluated for truthiness.
- **branch:** one possible block selected by a condition.
- **guard clause:** an early check that rejects or handles an invalid case.
- **exception:** an object representing a failure that interrupts normal flow.
- **raise:** to deliberately signal an exception.
- **handler:** an `except` block for a named exception type.

## Syntax anatomy

In `if score >= 90:`, the colon ends the condition and starts an
indented suite. Only code indented under that branch belongs to it. In
`except ValueError as exc:`, `ValueError` is the narrow failure type and
`exc` is the actual exception object. An exception handler is not a
substitute for an ordinary condition that can be checked directly.

### Worked example 1 — Order specific branches before general branches

Classify a value after validating its domain. Before running the next cell, predict its final displayed
value and identify the line responsible for every intermediate.

In [ ]:
def classify_score(score: int) -> str:
    if not 0 <= score <= 100:
        raise ValueError("score must be between 0 and 100")
    if score >= 90:
        return "distinction"
    if score >= 60:
        return "pass"
    return "fail"

[classify_score(value) for value in (42, 75, 95)]

**Expected observation:** `['fail', 'pass', 'distinction']`. The guard rejects out-of-domain values before classification.

If your result differs, compare inputs and types before rerunning.
Then explain the example from input to evidence in your own words.

### Worked example 2 — Protect only the fallible conversion

Let unrelated programming errors remain visible. Predict first; then run the next cell.

In [ ]:
def parse_count(text: str) -> int | None:
    try:
        count = int(text.strip())
    except ValueError:
        return None
    else:
        return count

[parse_count(text) for text in (" 12 ", "twelve")]

**Expected observation:** `[12, None]`. Only invalid integer text is converted into the chosen sentinel.

## Debugging clinic

When evidence differs from your prediction, use this order:

1. Trace conditions from top to bottom and mark the first one that becomes true.
2. Move validation before classification when impossible values enter normal branches.
3. Shrink a large `try` block until it contains only the operation with the expected failure.
4. Replace bare `except:` or `except Exception:` with the narrow type you can recover from.

**Alternative to compare:** Return a sentinel only when absence is part of the function's contract; otherwise raise a specific exception and let the caller decide.

**Boundary to test:** Boundary values such as 0, 1, 59, 60, 89, 90, and 100 expose gaps and overlaps in ordered conditions.

Do not move on merely because the cell runs. Explain which object,
branch, axis, row, or resource changed and why.

In [ ]:
def safe_divide(a: float, b: float) -> float:
    if b == 0:
        raise ZeroDivisionError('b must not be 0')
    return a / b

try:
    safe_divide(10, 0)
except ZeroDivisionError as e:
    print('Caught:', e)


## Custom exception
Create your own to signal domain-specific errors.

In [ ]:
class InvalidEmailError(ValueError):
    pass

def validate_email(email: str) -> None:
    if '@' not in email:
        raise InvalidEmailError('Missing @ in email')

for e in ['user@example.com', 'bad-email']:
    try:
        validate_email(e)
        print('ok:', e)
    except InvalidEmailError as err:
        print('invalid:', err)


## Exercises and progressive hints

Each item is a complete mini-contract. Before writing code, copy its input,
expected behavior, constraints, and verification into your work cell. A
result is not complete merely because it “looks right”; run the stated
assertion or inspection and explain what it proves.

1. Print FizzBuzz for every integer from 1 through 30 inclusive. **Rules:** multiples of both 3 and 5 print `FizzBuzz`; only 3 prints `Fizz`; only 5 prints `Buzz`; all others print the number. **Constraints:** use one ordered `if`/`elif`/`else` chain.
   **Verify:** capture the output lines, assert there are exactly 30, and assert positions 3, 5, 15, 16, and 30 equal `Fizz`, `Buzz`, `FizzBuzz`, `16`, and `FizzBuzz`.

2. Write `parse_age(text)` that returns an integer for valid integer text and returns `None` only when conversion raises `ValueError`. **Constraints:** put only `int(text)` inside `try`; do not use a bare `except`.
   **Verify:** test `'42'`, `' 7 '`, and `'seven'`, then explain which path uses `else`.

3. Define `NegativeMeasurementError` as a subclass of `ValueError`, then write `validate_measurement(value)` that returns non-negative values unchanged and raises your exception for `-0.1`.
   **Verify:** show one normal return and catch the exact custom type in a small demonstration; do not catch it inside the validator.

### Additional mastery practice

Make branch order and exception boundaries deliberate. Catch only the failure that the current layer can interpret or repair.

Continue with five new exercises. Record each prediction before running
code; these extend rather than replace the original practice above.

4. **Prediction:** Predict which branch handles 0, 9, and 12 when checks are ordered as `x % 2 == 0`, `x % 3 == 0`, then both. Explain the bug.
   **Progressive hint:** Only the first true branch runs; test the most specific rule first.
   **Verify:** Record the branch selected for `0`, `9`, and `12` before/after reordering; confirm the repaired `12` reaches the combined rule.
5. **Tracing:** Trace `try/except/else/finally` for a successful integer parse and for invalid text. Which clauses run in each path?
   **Progressive hint:** `else` follows success; `finally` runs in both cases.
   **Verify:** Use a four-column trace for `try`, `except`, `else`, and `finally` on valid/invalid text; each clause's executed flag must match the language rules.
6. **Implementation:** Implement `classify_score(score)` returning fail/pass/distinction and reject scores outside 0–100 with `ValueError`.
   **Progressive hint:** Validate the domain before choosing a result branch.
   **Verify:** Assert representative scores return `fail`, `pass`, and `distinction`, then assert `-1` and `101` each raise `ValueError`.
7. **Debugging:** Replace a bare `except:` around parsing and calculation with the narrowest useful handler, while allowing programming errors to surface.
   **Progressive hint:** Keep only the conversion inside the protected block.
   **Verify:** Show invalid integer text is handled while a deliberate unrelated `TypeError` still reaches the test; keep only conversion inside `try`.
8. **Edge case and explanation:** Design `safe_ratio(numerator, denominator)` for a zero denominator. Choose between raising, returning `None`, or a default and justify it.
   **Progressive hint:** A reusable library function usually should not invent a numeric result.
   **Verify:** Test a nonzero ratio and a zero denominator; assert the chosen zero policy exactly and explain why no fabricated numeric answer leaks through.

Before opening the reference solution, write one sentence explaining
which contract or mental model each result confirms.

### Practice 1 — prediction, attempt, and evidence

**Contract reminder:** Print FizzBuzz for every integer from 1 through 30 inclusive. **Rules:** multiples of both 3 and 5 print `FizzBuzz`; only 3 prints `Fizz`; only 5 prints `Buzz`; all others print the number. **Constraints:** use one ordered `if`/`elif`/`else` chain. **Verify:** capture the output lines, assert there are exactly 30, and assert positions 3, 5, 15, 16, and 30 equal `Fizz`, `Buzz`, `FizzBuzz`, `16`, and `FizzBuzz`.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 1 — your work
# Short contract: Print FizzBuzz for every integer from 1 through 30 inclusive. multiples of both 3 and 5 print `FizzBuzz`; only 3 prints `Fizz`; only 5 prints `Buzz`; all others print the number...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 2 — prediction, attempt, and evidence

**Contract reminder:** Write `parse_age(text)` that returns an integer for valid integer text and returns `None` only when conversion raises `ValueError`. **Constraints:** put only `int(text)` inside `try`; do not use a bare `except`. **Verify:** test `'42'`, `' 7 '`, and `'seven'`, then explain which path uses `else`.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 2 — your work
# Short contract: Write `parse_age(text)` that returns an integer for valid integer text and returns `None` only when conversion raises `ValueError`. put only `int(text)` inside `try`; do not use...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 3 — prediction, attempt, and evidence

**Contract reminder:** Define `NegativeMeasurementError` as a subclass of `ValueError`, then write `validate_measurement(value)` that returns non-negative values unchanged and raises your exception for `-0.1`. **Verify:** show one normal return and catch the exact custom type in a small demonstration; do not catch it inside the validator.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 3 — your work
# Short contract: Define `NegativeMeasurementError` as a subclass of `ValueError`, then write `validate_measurement(value)` that returns non-negative values unchanged and raises your exception fo...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 4 — prediction, attempt, and evidence

**Contract reminder:** **Prediction:** Predict which branch handles 0, 9, and 12 when checks are ordered as `x % 2 == 0`, `x % 3 == 0`, then both. Explain the bug. **Progressive hint:** Only the first true branch runs; test the most specific rule first. **Verify:** Record the branch selected for `0`, `9`, and `12` before/after reordering; confirm the repaired `12` reaches the combined rule.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 4 — your work
# Short contract: Predict which branch handles 0, 9, and 12 when checks are ordered as `x % 2 == 0`, `x % 3 == 0`, then both. Explain the bug. Only the first true branch runs; test the most speci...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 5 — prediction, attempt, and evidence

**Contract reminder:** **Tracing:** Trace `try/except/else/finally` for a successful integer parse and for invalid text. Which clauses run in each path? **Progressive hint:** `else` follows success; `finally` runs in both cases. **Verify:** Use a four-column trace for `try`, `except`, `else`, and `finally` on valid/invalid text; each clause's executed flag must match the language rules.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 5 — your work
# Short contract: Trace `try/except/else/finally` for a successful integer parse and for invalid text. Which clauses run in each path? `else` follows success; `finally` runs in both cases. Use a...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 6 — prediction, attempt, and evidence

**Contract reminder:** **Implementation:** Implement `classify_score(score)` returning fail/pass/distinction and reject scores outside 0–100 with `ValueError`. **Progressive hint:** Validate the domain before choosing a result branch. **Verify:** Assert representative scores return `fail`, `pass`, and `distinction`, then assert `-1` and `101` each raise `ValueError`.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 6 — your work
# Short contract: Implement `classify_score(score)` returning fail/pass/distinction and reject scores outside 0–100 with `ValueError`. Validate the domain before choosing a result branch. Assert...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 7 — prediction, attempt, and evidence

**Contract reminder:** **Debugging:** Replace a bare `except:` around parsing and calculation with the narrowest useful handler, while allowing programming errors to surface. **Progressive hint:** Keep only the conversion inside the protected block. **Verify:** Show invalid integer text is handled while a deliberate unrelated `TypeError` still reaches the test; keep only conversion inside `try`.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 7 — your work
# Short contract: Replace a bare `except:` around parsing and calculation with the narrowest useful handler, while allowing programming errors to surface. Keep only the conversion inside the prot...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):


### Practice 8 — prediction, attempt, and evidence

**Contract reminder:** **Edge case and explanation:** Design `safe_ratio(numerator, denominator)` for a zero denominator. Choose between raising, returning `None`, or a default and justify it. **Progressive hint:** A reusable library function usually should not invent a numeric result. **Verify:** Test a nonzero ratio and a zero denominator; assert the chosen zero policy exactly and explain why no fabricated numeric answer leaks through.

In the next cell, record your prediction before the code. Keep
the input tiny, implement only this contract, and finish with
the requested assertion or bounded inspection. If it fails,
retain the smallest failing input and write what the evidence
changed about your hypothesis.

In [ ]:
# Practice 8 — your work
# Short contract: Design `safe_ratio(numerator, denominator)` for a zero denominator. Choose between raising, returning `None`, or a default and justify it. A reusable library function usually sh...
# Prediction:

# Implementation:

# Verification (assert or inspect exactly what the prompt requires):
